In [2]:
import pandas as pd
import polars as pl
import numpy as np
import random

# Config
DATA_PATH = "C:/Uner/Semester 5/Proses Stokastik/Coolyeah/UAS/sampled_data.csv"
SAMPLE_N = 2000
MAP_NAME = "ERANGEL"
DT_PHASE = 300.0   # phase length in seconds
DT_EVAL = 60.0     # evaluation grid for N(t)

In [3]:
# Load once (Polars), normalize "map", sample match_ids and convert to pandas
pl_df = pl.read_csv(DATA_PATH)
# make an uppercase map column to compare robustly
if "map" in pl_df.columns:
    pl_df = pl_df.with_columns(pl.col("map").str.to_uppercase().alias("map_uc"))
else:
    pl_df = pl_df.with_columns(pl.lit("UNKNOWN").alias("map_uc"))

one_map_df = pl_df.filter(pl.col("map_uc") == MAP_NAME)
match_ids = one_map_df.select("match_id").unique().to_series().to_list()
n_sample = min(SAMPLE_N, len(match_ids))
sampled_match_ids = random.sample(match_ids, n_sample) if n_sample > 0 else []
sampled_data_pl = one_map_df.filter(pl.col("match_id").is_in(sampled_match_ids))

# Convert to pandas and prepare time/map columns
df = sampled_data_pl.to_pandas() if sampled_data_pl.height > 0 else pd.DataFrame()
if not df.empty:
    if "map" in df.columns:
        df["map"] = df["map"].astype(str).str.upper()
    else:
        df["map"] = MAP_NAME
    df["time"] = pd.to_numeric(df.get("time", pd.Series([])), errors="coerce")
    df = df.dropna(subset=["time"]).reset_index(drop=True)

df_erangel = df[df["map"] == MAP_NAME].copy()

In [4]:
# Descriptive statistics for times (single place)
times = df_erangel["time"].astype(float) if not df_erangel.empty else pd.Series(dtype=float)
n = int(times.shape[0])
t_min = float(times.min()) if n>0 else float("nan")
t_max = float(times.max()) if n>0 else float("nan")
t_mean = float(times.mean()) if n>0 else float("nan")
t_median = float(times.median()) if n>0 else float("nan")
t_std = float(times.std()) if n>0 else float("nan")
q1 = float(times.quantile(0.25)) if n>0 else float("nan")
q3 = float(times.quantile(0.75)) if n>0 else float("nan")

sorted_times = np.sort(times.values) if n>0 else np.array([])
gaps = np.diff(sorted_times) if sorted_times.size > 1 else np.array([])
gap_mean = float(gaps.mean()) if gaps.size>0 else float("nan")
gap_median = float(np.median(gaps)) if gaps.size>0 else float("nan")

analysis = (
    f"Analisis deskriptif waktu kematian di map {MAP_NAME}:\n"
    f"- Total kematian yang tercatat: {n} event.\n"
    f"- Waktu kematian paling awal: {t_min:.1f}, paling akhir: {t_max:.1f} detik.\n"
    f"- Mean: {t_mean:.1f} detik, median: {t_median:.1f} detik, std: {t_std:.1f} detik.\n"
    f"- IQR: {q3 - q1:.1f} detik (Q1={q1:.1f}, Q3={q3:.1f}).\n"
    f"- Interarrival mean: {gap_mean:.2f} detik, median: {gap_median:.2f} detik.\n"
)

print(analysis)

Analisis deskriptif waktu kematian di map ERANGEL:
- Total kematian yang tercatat: 183236 event.
- Waktu kematian paling awal: 63.0, paling akhir: 2211.0 detik.
- Mean: 762.8 detik, median: 626.0 detik, std: 557.7 detik.
- IQR: 1008.0 detik (Q1=243.0, Q3=1251.0).
- Interarrival mean: 0.01 detik, median: 0.00 detik.



In [5]:
# Phase-based rate estimation (single computation)
n_match = int(df_erangel["match_id"].nunique()) if not df_erangel.empty else 0
T_global = float(df_erangel["time"].max()) if not df_erangel.empty else 0.0
n_phase = int(np.ceil(T_global / DT_PHASE)) if T_global > 0 else 0

if n_phase > 0:
    df_erangel["phase"] = (df_erangel["time"] // DT_PHASE).astype(int)
    phase_counts = (
        df_erangel.groupby("phase").size().rename("n_events").reset_index()
    )
    phase_counts["T_phase"] = DT_PHASE * n_match
    phase_counts["lambda_hat"] = phase_counts["n_events"] / phase_counts["T_phase"]
else:
    phase_counts = pd.DataFrame(columns=["phase","n_events","T_phase","lambda_hat"])

print(phase_counts)

   phase  n_events   T_phase  lambda_hat
0      0     56529  600000.0    0.094215
1      1     33207  600000.0    0.055345
2      2     23745  600000.0    0.039575
3      3     20679  600000.0    0.034465
4      4     22244  600000.0    0.037073
5      5     21579  600000.0    0.035965
6      6      5173  600000.0    0.008622
7      7        80  600000.0    0.000133


In [6]:
# Build cumulative intensity and global N_hat function once
lambda_arr = phase_counts.sort_values("phase")["lambda_hat"].to_numpy() if not phase_counts.empty else np.array([])
n_phase = len(lambda_arr)
cum_intensity_phase = np.concatenate(([0.0], np.cumsum(lambda_arr * DT_PHASE))) if n_phase>0 else np.array([0.0])

def N_hat_global(t):
    """Expected cumulative deaths per match up to time t according to NHPP with piecewise-constant rates."""
    t = float(t)
    if t <= 0 or n_phase == 0:
        return 0.0
    k = int(t // DT_PHASE)
    if k >= n_phase:
        k = n_phase - 1
        t_in_phase = DT_PHASE
    else:
        t_in_phase = t - k * DT_PHASE
    base = cum_intensity_phase[k]
    lam_k = lambda_arr[k] if k < n_phase else (lambda_arr[-1] if n_phase>0 else 0.0)
    return base + lam_k * t_in_phase

# Comparison helper and full-run
def compare_one_match(mid, group, dt_eval=DT_EVAL):
    times = np.sort(group["time"].values.astype(float)) if not group.empty else np.array([])
    T_max = times.max() if times.size>0 else 0.0
    t_grid = np.arange(0, T_max + dt_eval, dt_eval)
    N_data = np.searchsorted(times, t_grid, side="right")
    N_hat = np.array([N_hat_global(t) for t in t_grid])
    return pd.DataFrame({
        "match_id": mid,
        "time": t_grid,
        "N_data": N_data,
        "N_hat": N_hat,
    })

all_comp = [compare_one_match(mid, g) for mid, g in df_erangel.groupby("match_id")] if not df_erangel.empty else []
comparison_all = pd.concat(all_comp, ignore_index=True) if all_comp else pd.DataFrame(columns=["match_id","time","N_data","N_hat"])

print(comparison_all.head())

                                            match_id   time  N_data    N_hat
0  2U4GBNA0Ymk-5S79f_s6_VaeWn4TbKjY-mPjjyW2FLsPYJ...    0.0       0   0.0000
1  2U4GBNA0Ymk-5S79f_s6_VaeWn4TbKjY-mPjjyW2FLsPYJ...   60.0       0   5.6529
2  2U4GBNA0Ymk-5S79f_s6_VaeWn4TbKjY-mPjjyW2FLsPYJ...  120.0       4  11.3058
3  2U4GBNA0Ymk-5S79f_s6_VaeWn4TbKjY-mPjjyW2FLsPYJ...  180.0      15  16.9587
4  2U4GBNA0Ymk-5S79f_s6_VaeWn4TbKjY-mPjjyW2FLsPYJ...  240.0      24  22.6116
